In [1]:
import numpy as np
import pandas as pd


In [7]:
df = pd.read_csv("Core_audit_start.csv")
df.head(2)

,Assigned country,Country resource level,Dataset already in the matrix?,Dataset name,Task Category,Country Attribution Method,Year created,Languages covered (separate using semicolon),Language coverage type,Dataset Link (URL),...,canonical_language_list_cleaned,variant_language_list_cleaned,sign_language_list_cleaned,excluded_language_like_tokens_cleaned,language_type_records_cleaned,canonical_language_count_cleaned,variant_language_count_cleaned,sign_language_count_cleaned,excluded_language_like_token_count_cleaned,canonical_language_string_cleaned
0,NaN,NaN,NaN,Not stated,"Human-AI Interaction / Cooperation, Human-Cent...",Not stated,2025,Not stated,Monolingual,Not stated,...,[],[],[],[],[],0,0,0,0,NaN
1,NaN,NaN,NaN,Wiki-NFQA,Question Answering,Not stated,2025,English,Monolingual,Not stated,...,"[""English""]",[],[],[],"[{""raw_language_token"": ""English"", ""normalized...",1,0,0,0,English


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18035 entries, 0 to 18034
Data columns (total 70 columns):
 #   Column                                                                Non-Null Count  Dtype  
---  ------                                                                --------------  -----  
 0   Assigned country                                                      0 non-null      float64
 1   Country resource level                                                0 non-null      float64
 2   Dataset already in the matrix?                                        0 non-null      float64
 3   Dataset name                                                          18035 non-null  object 
 4   Task Category                                                         18035 non-null  object 
 5   Country Attribution Method                                            18035 non-null  object 
 6   Year created                                                          18035 non-null  int64  


In [9]:
import pandas as pd
import numpy as np
import ast

# ============================================================
# STAGE 0: BUILD ATLASNLP AUDIT SAMPLING FRAME
# ============================================================

# Work from a copy so the 70-column master dataframe is untouched
master = df.copy()

# ------------------------------------------------------------
# 1. Helper: safely convert list-like country fields into sets
# ------------------------------------------------------------

def parse_country_set(value):
    """
    Convert AtlasNLP country fields into a clean Python set.

    Handles:
      - actual Python lists
      - strings that look like lists, e.g. "['India', 'Nigeria']"
      - semicolon-separated strings
      - comma-separated strings
      - empty lists / missing values
    """

    if value is None:
        return set()

    # NaN
    if isinstance(value, float) and pd.isna(value):
        return set()

    # Already list-like
    if isinstance(value, (list, tuple, set)):
        return {
            str(x).strip()
            for x in value
            if str(x).strip()
        }

    value = str(value).strip()

    if value in {"", "[]", "nan", "None", "Not stated"}:
        return set()

    # Try parsing Python-list representation first
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, (list, tuple, set)):
                return {
                    str(x).strip()
                    for x in parsed
                    if str(x).strip()
                }
        except (ValueError, SyntaxError):
            pass

    # Fallback: semicolon-separated
    if ";" in value:
        return {
            x.strip()
            for x in value.split(";")
            if x.strip()
        }

    # Fallback: comma-separated
    if "," in value:
        return {
            x.strip()
            for x in value.split(",")
            if x.strip()
        }

    # Single country
    return {value}


# ------------------------------------------------------------
# 2. Parse authoritative content and producer country fields
# ------------------------------------------------------------

master["_content_country_set"] = (
    master["best_country_list_valid"]
    .apply(parse_country_set)
)

master["_producer_country_set"] = (
    master["producer_country_list"]
    .apply(parse_country_set)
)


# ------------------------------------------------------------
# 3. Define producer/content relationship
# ------------------------------------------------------------

def classify_relationship(row):

    content = row["_content_country_set"]
    producer = row["_producer_country_set"]

    if len(content) == 0 and len(producer) == 0:
        return "BOTH_MISSING"

    if len(content) == 0:
        return "CONTENT_MISSING"

    if len(producer) == 0:
        return "PRODUCER_MISSING"

    if content == producer:
        return "EXACT"

    if content & producer:
        return "PARTIAL"

    return "NONE"


master["producer_content_relationship"] = master.apply(
    classify_relationship,
    axis=1
)


# ------------------------------------------------------------
# 4. Multi-country content flag
# ------------------------------------------------------------

master["multi_country_content"] = (
    master["_content_country_set"].apply(len) >= 2
)


# ------------------------------------------------------------
# 5. Create permanent audit ID
# ------------------------------------------------------------

# source_row_index appears to run from 0 to 18034.
# Adding 1 gives human-friendly IDs WB000001 ... WB018035.

master["audit_record_id"] = (
    "WB"
    + (master["source_row_index"] + 1)
      .astype(int)
      .astype(str)
      .str.zfill(6)
)


# ------------------------------------------------------------
# 6. Generate ONE reproducible sampling random number per record
# ------------------------------------------------------------

RANDOM_SEED = 20260820

rng = np.random.default_rng(RANDOM_SEED)

master["sampling_random_number"] = rng.random(len(master))


# ------------------------------------------------------------
# 7. Select columns for Stage 0 audit frame
# ------------------------------------------------------------

audit_columns = [
    "audit_record_id",
    "source_row_index",

    # paper/dataset identity
    "paper_url",
    "paper_url_key",
    "Dataset name",
    "Dataset Link (URL)",

    # task/language
    "normalized_task_category",
    "has_valid_task",
    "canonical_language_string_cleaned",
    "canonical_language_count_cleaned",

    # content geography
    "best_country_field_used",
    "best_country_list_raw",
    "best_country_list_valid",
    "has_valid_content_country",
    "num_content_countries",

    # producer geography
    "producer_country_list",
    "producer_country_source_used",
    "has_producer_country",
    "num_producer_countries",

    # NEW Stage 0 variables
    "producer_content_relationship",
    "multi_country_content",
    "sampling_random_number",
]

audit_frame = master[audit_columns].copy()


# ------------------------------------------------------------
# 8. Sanity checks
# ------------------------------------------------------------

assert len(audit_frame) == 18035, (
    f"Expected 18,035 records, found {len(audit_frame):,}"
)

assert audit_frame["audit_record_id"].is_unique, (
    "audit_record_id is not unique"
)

assert audit_frame["source_row_index"].is_unique, (
    "source_row_index is not unique"
)

assert audit_frame["paper_url_key"].is_unique, (
    "paper_url_key is not unique. "
    "Check whether there are repeated paper URLs."
)


# Check that our parsed country counts agree with the existing pipeline counts
parsed_content_counts = master["_content_country_set"].apply(len)

content_count_mismatches = (
    parsed_content_counts
    != master["num_content_countries"]
).sum()

print(
    "Content-country count mismatches:",
    content_count_mismatches
)


# ------------------------------------------------------------
# 9. Show Stage 0 population counts
# ------------------------------------------------------------

print("\n=== TOTAL AUDIT FRAME ===")
print(f"{len(audit_frame):,} records")


print("\n=== CONTENT COUNTRY PRESENT ===")
print(
    audit_frame["has_valid_content_country"]
    .value_counts(dropna=False)
)


print("\n=== PRODUCER / CONTENT RELATIONSHIP ===")
relationship_counts = (
    audit_frame["producer_content_relationship"]
    .value_counts()
    .rename_axis("relationship")
    .reset_index(name="n_records")
)

relationship_counts["percent"] = (
    relationship_counts["n_records"]
    / len(audit_frame)
    * 100
).round(2)

print(relationship_counts.to_string(index=False))


print("\n=== MULTI-COUNTRY CONTENT ===")
print(
    audit_frame["multi_country_content"]
    .value_counts(dropna=False)
)


print("\n=== CONTENT COUNTRY COUNTS ===")
print(
    audit_frame["num_content_countries"]
    .value_counts()
    .sort_index()
    .head(20)
)


# ------------------------------------------------------------
# 10. Useful cross-tab for sampling design
# ------------------------------------------------------------

relationship_by_multi = pd.crosstab(
    audit_frame["producer_content_relationship"],
    audit_frame["multi_country_content"],
    margins=True
)

print("\n=== RELATIONSHIP × MULTI-COUNTRY ===")
print(relationship_by_multi)


# ------------------------------------------------------------
# 11. Save Stage 0 outputs
# ------------------------------------------------------------

audit_frame.to_csv(
    "atlasnlp_audit_sampling_frame.csv",
    index=False
)

relationship_counts.to_csv(
    "atlasnlp_audit_stratum_counts.csv",
    index=False
)

print("\nSaved:")
print("  atlasnlp_audit_sampling_frame.csv")
print("  atlasnlp_audit_stratum_counts.csv")

Content-country count mismatches: 0

=== TOTAL AUDIT FRAME ===
18,035 records

=== CONTENT COUNTRY PRESENT ===
True     11764
False     6271
Name: has_valid_content_country, dtype: int64

=== PRODUCER / CONTENT RELATIONSHIP ===
    relationship  n_records  percent
           EXACT       6845    37.95
 CONTENT_MISSING       6117    33.92
         PARTIAL       2817    15.62
            NONE       1890    10.48
PRODUCER_MISSING        212     1.18
    BOTH_MISSING        154     0.85

=== MULTI-COUNTRY CONTENT ===
False    14927
True      3108
Name: multi_country_content, dtype: int64

=== CONTENT COUNTRY COUNTS ===
0     6271
1     8656
2     1662
3      545
4      247
5      173
6      101
7       72
8       49
9       46
10      34
11      28
12      18
13      15
14      12
15       9
16       7
17       7
18       8
19      12
Name: num_content_countries, dtype: int64

=== RELATIONSHIP × MULTI-COUNTRY ===
multi_country_content          False  True    All
producer_content_relationshi

## Pick 400 random rows

In [10]:
# ============================================================
# STAGE 1: DRAW REPRESENTATIVE RANDOM SAMPLE OF 400
# Population: records with a valid content-country assignment
# ============================================================

# 1. Define Stage 1 population
stage1_population = audit_frame[
    audit_frame["has_valid_content_country"] == True
].copy()

print(f"Stage 1 population: {len(stage1_population):,} records")

assert len(stage1_population) == 11764


# ------------------------------------------------------------
# 2. Draw the first 400 according to the fixed random ordering
# ------------------------------------------------------------

stage1_sample = (
    stage1_population
    .sort_values("sampling_random_number")
    .head(400)
    .copy()
)

assert len(stage1_sample) == 400
assert stage1_sample["audit_record_id"].is_unique


# ------------------------------------------------------------
# 3. Add sampling metadata
# ------------------------------------------------------------

stage1_sample["audit_stage"] = "S1_RANDOM_COUNTRY_PRESENT"

# Give the records a convenient audit order from 1–400.
# This is the order you can use in the Google Sheet.
stage1_sample["audit_order"] = range(1, 401)


# ------------------------------------------------------------
# 4. Add EMPTY manual annotation columns
# ------------------------------------------------------------

manual_columns = [
    # Dataset/source verification
    "dataset_relationship",
    # INTRODUCED_HERE / EXTENDED_HERE / REUSED_EXISTING /
    # MULTIPLE_DATASETS_OR_AMBIGUOUS

    # Content-country audit
    "manual_content_countries",
    "content_country_verdict",
    # EXACT / PARTIAL / INCORRECT / CANNOT_DETERMINE

    "country_set_error_type",
    # NONE / OVERATTRIBUTION / UNDERATTRIBUTION /
    # BOTH_OVER_AND_UNDER / WRONG_COUNTRY_SET / UNDETERMINED

    "content_evidence_type",
    # EXPLICIT_DATASET_PROVENANCE / PARTICIPANT_LOCATION /
    # COUNTRY_SPECIFIC_SOURCE / COUNTRY_SPLIT_TABLE /
    # REPRESENTED_POPULATION / GEOGRAPHIC_COLLECTION_FILTER /
    # OTHER_STRONG_EVIDENCE / NO_SUPPORT / AMBIGUOUS

    "evidence_source",
    # CURRENT_PAPER / TABLE_OR_FIGURE / APPENDIX /
    # ORIGINAL_DATASET_PAPER / DATASET_CARD_OR_REPOSITORY /
    # MULTIPLE_SOURCES

    "content_evidence_page",
    "content_evidence_note",

    "independent_content_evidence",
    # YES / NO / AMBIGUOUS

    "affiliation_leakage",
    # YES / NO / POSSIBLE / NOT_APPLICABLE

    "failure_code",

    # Other AtlasNLP fields that can be checked while paper is open
    "manual_producer_countries",
    "producer_country_verdict",

    "manual_task",
    "task_verdict",

    "manual_languages",
    "language_verdict",

    # General notes
    "annotator_notes",
]

for col in manual_columns:
    stage1_sample[col] = ""


# ------------------------------------------------------------
# 5. Put columns in a convenient inspection order
# ------------------------------------------------------------

inspection_columns = [
    # Audit bookkeeping
    "audit_order",
    "audit_record_id",
    "audit_stage",

    # Source
    "paper_url",
    "Dataset name",
    "Dataset Link (URL)",

    # AtlasNLP content geography
    "best_country_list_valid",
    "num_content_countries",
    "best_country_field_used",

    # Producer geography
    "producer_country_list",
    "num_producer_countries",
    "producer_content_relationship",

    # Useful context
    "multi_country_content",
    "normalized_task_category",
    "canonical_language_string_cleaned",

    # ------------------------
    # Manual annotation fields
    # ------------------------
    "dataset_relationship",

    "manual_content_countries",
    "content_country_verdict",
    "country_set_error_type",

    "content_evidence_type",
    "evidence_source",
    "content_evidence_page",
    "content_evidence_note",

    "independent_content_evidence",
    "affiliation_leakage",
    "failure_code",

    "manual_producer_countries",
    "producer_country_verdict",

    "manual_task",
    "task_verdict",

    "manual_languages",
    "language_verdict",

    "annotator_notes",

    # Keep traceability / sampling info at the end
    "source_row_index",
    "paper_url_key",
    "sampling_random_number",
]

stage1_sample = stage1_sample[inspection_columns]


# ------------------------------------------------------------
# 6. Inspect the composition of the random sample
# ------------------------------------------------------------

print("\n=== PRODUCER / CONTENT RELATIONSHIP ===")
print(
    stage1_sample["producer_content_relationship"]
    .value_counts()
)


print("\n=== MULTI-COUNTRY CONTENT ===")
print(
    stage1_sample["multi_country_content"]
    .value_counts()
)


print("\n=== NUMBER OF CONTENT COUNTRIES ===")
print(
    stage1_sample["num_content_countries"]
    .value_counts()
    .sort_index()
)


# Useful combined table
print("\n=== RELATIONSHIP × MULTI-COUNTRY ===")
print(
    pd.crosstab(
        stage1_sample["producer_content_relationship"],
        stage1_sample["multi_country_content"]
    )
)


# ------------------------------------------------------------
# 7. Save the manual audit sheet
# ------------------------------------------------------------

stage1_sample.to_csv(
    "atlasnlp_stage1_random400_preaudit.csv",
    index=False
)

print("\nSaved: atlasnlp_stage1_random400_preaudit.csv")

Stage 1 population: 11,764 records

=== PRODUCER / CONTENT RELATIONSHIP ===
EXACT               241
PARTIAL              87
NONE                 68
PRODUCER_MISSING      4
Name: producer_content_relationship, dtype: int64

=== MULTI-COUNTRY CONTENT ===
False    299
True     101
Name: multi_country_content, dtype: int64

=== NUMBER OF CONTENT COUNTRIES ===
1     299
2      60
3      15
4      12
5       4
6       3
12      2
14      2
21      1
47      1
74      1
Name: num_content_countries, dtype: int64

=== RELATIONSHIP × MULTI-COUNTRY ===
multi_country_content          False  True 
producer_content_relationship              
EXACT                            216     25
NONE                              48     20
PARTIAL                           31     56
PRODUCER_MISSING                   4      0

Saved: atlasnlp_stage1_random400_audit.csv


## Audit analysis

In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
from math import sqrt

PATH = "atlasnlp_stage1_random400_ai_audit_completed_400.csv"

df = pd.read_csv(PATH)

# ============================================================
# 1. Integrity checks
# ============================================================

assert len(df) == 400
assert df["audit_order"].nunique() == 400
assert df["audit_record_id"].nunique() == 400
assert df["source_row_index"].nunique() == 400
assert df["paper_url_key"].nunique() == 400

assert df["human_content_country_verdict"].notna().all()
assert df["human_failure_code"].notna().all()
assert df["human_producer_country_verdict"].notna().all()

print("Audit file passed integrity checks.")


# ============================================================
# 2. Wilson 95% confidence intervals
# ============================================================

def wilson_ci(k, n, z=1.96):
    p = k / n

    denominator = 1 + z**2 / n

    center = (
        p + z**2 / (2*n)
    ) / denominator

    half_width = (
        z
        * np.sqrt(
            p*(1-p)/n
            + z**2/(4*n**2)
        )
        / denominator
    )

    return p, center-half_width, center+half_width


verdict_counts = (
    df["human_content_country_verdict"]
    .value_counts()
)

print("\n=== OVERALL CONTENT-COUNTRY VERDICT ===")

for verdict in ["Correct", "Inferred", "Incorrect"]:

    k = verdict_counts.get(verdict, 0)

    p, low, high = wilson_ci(k, len(df))

    print(
        f"{verdict:10s}: "
        f"{k:3d}/400 = {p*100:5.1f}% "
        f"(95% CI {low*100:.1f}–{high*100:.1f}%)"
    )


supported = (
    df["human_content_country_verdict"]
    .isin(["Correct", "Inferred"])
    .sum()
)

p, low, high = wilson_ci(supported, len(df))

print(
    f"\nCorrect + Inferred: "
    f"{supported}/400 = {p*100:.1f}% "
    f"(95% CI {low*100:.1f}–{high*100:.1f}%)"
)


# ============================================================
# 3. Producer/content relationship
# ============================================================

relationship_table = pd.crosstab(
    df["producer_content_relationship"],
    df["human_content_country_verdict"]
)

relationship_percent = (
    relationship_table
    .div(relationship_table.sum(axis=1), axis=0)
    * 100
).round(1)

print("\n=== COUNTS BY PRODUCER/CONTENT RELATIONSHIP ===")
print(relationship_table)

print("\n=== PERCENT BY PRODUCER/CONTENT RELATIONSHIP ===")
print(relationship_percent)


# ============================================================
# 4. Single vs multi-country
# ============================================================

multi_table = pd.crosstab(
    df["multi_country_content"],
    df["human_content_country_verdict"]
)

multi_percent = (
    multi_table
    .div(multi_table.sum(axis=1), axis=0)
    * 100
).round(1)

print("\n=== SINGLE VS MULTI-COUNTRY COUNTS ===")
print(multi_table)

print("\n=== SINGLE VS MULTI-COUNTRY PERCENT ===")
print(multi_percent)


# ============================================================
# 5. Number of content countries
# ============================================================

country_count_table = pd.crosstab(
    df["num_content_countries"],
    df["human_content_country_verdict"]
)

country_count_percent = (
    country_count_table
    .div(country_count_table.sum(axis=1), axis=0)
    * 100
).round(1)

print("\n=== VERDICT BY NUMBER OF CONTENT COUNTRIES ===")
print(country_count_table)

print("\n=== PERCENT BY NUMBER OF CONTENT COUNTRIES ===")
print(country_count_percent)


# ============================================================
# 6. Parse failure codes
# ============================================================

def parse_failure_codes(value):

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    # Most rows use semicolons; a few use commas.
    pieces = re.split(r"[;,]", value)

    return [
        piece.strip().upper().replace(" ", "_")
        for piece in pieces
        if piece.strip()
    ]


failure_counter = Counter()

for value in df["human_failure_code"]:
    failure_counter.update(parse_failure_codes(value))


failure_table = pd.DataFrame(
    failure_counter.most_common(),
    columns=["failure_code", "records"]
)

failure_table["percent_of_400"] = (
    failure_table["records"] / 400 * 100
).round(1)

print("\n=== FAILURE MECHANISMS ===")
print(failure_table.head(25).to_string(index=False))


# ============================================================
# 7. Affiliation leakage
# ============================================================

affiliation_leakage = (
    df["human_failure_code"]
    .apply(
        lambda x:
        "AFFIL_AS_CONTENT" in parse_failure_codes(x)
    )
)

print("\n=== AFFILIATION LEAKAGE ===")

print(
    f"Overall: "
    f"{affiliation_leakage.sum()}/400 "
    f"= {affiliation_leakage.mean()*100:.1f}%"
)

incorrect_mask = (
    df["human_content_country_verdict"]
    == "Incorrect"
)

print(
    f"Among Incorrect: "
    f"{(affiliation_leakage & incorrect_mask).sum()}/"
    f"{incorrect_mask.sum()} "
    f"= "
    f"{(affiliation_leakage & incorrect_mask).sum()/incorrect_mask.sum()*100:.1f}%"
)

affil_by_relationship = pd.crosstab(
    df["producer_content_relationship"],
    affiliation_leakage
)

print("\nAffiliation leakage by relationship:")
print(affil_by_relationship)

print(
    "\nPercent within relationship:"
)

print(
    (
        affil_by_relationship
        .div(affil_by_relationship.sum(axis=1), axis=0)
        * 100
    ).round(1)
)


# ============================================================
# 8. Compare Atlas country set with audited proposed set
# ============================================================

def parse_country_list(value):

    if pd.isna(value):
        return set()

    if isinstance(value, (list, tuple, set)):
        return {
            str(x).strip()
            for x in value
            if str(x).strip()
        }

    value = str(value).strip()

    if value in {
        "",
        "[]",
        "None",
        "nan",
        "Not stated"
    }:
        return set()

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, (list, tuple, set)):
            return {
                str(x).strip()
                for x in parsed
                if str(x).strip()
            }

    except (ValueError, SyntaxError):
        pass

    if ";" in value:
        return {
            x.strip()
            for x in value.split(";")
            if x.strip()
        }

    return {value}


atlas_sets = (
    df["best_country_list_valid"]
    .apply(parse_country_list)
)

audit_sets = (
    df["ai_manual_content_countries"]
    .apply(parse_country_list)
)


def compare_country_sets(atlas_set, audit_set):

    if atlas_set == audit_set:
        return "EXACT_SAME"

    if not audit_set:
        return "MANUAL_EMPTY"

    if not atlas_set:
        return "OLD_EMPTY"

    if not atlas_set.intersection(audit_set):
        return "DISJOINT"

    if audit_set < atlas_set:
        return "OLD_OVERATTRIBUTED"

    if atlas_set < audit_set:
        return "OLD_UNDERATTRIBUTED"

    return "BOTH_OVER_AND_UNDER"


df["country_set_change"] = [
    compare_country_sets(old, new)
    for old, new in zip(atlas_sets, audit_sets)
]

print("\n=== TYPE OF COUNTRY-SET CORRECTION ===")

print(
    pd.crosstab(
        df["human_content_country_verdict"],
        df["country_set_change"]
    )
)


# ============================================================
# 9. Language-group exploratory analysis
# ============================================================

def language_group(value):

    if pd.isna(value) or not str(value).strip():
        return "LANGUAGE_MISSING"

    languages = [
        x.strip()
        for x in str(value).split(";")
        if x.strip()
    ]

    lowercase = [x.lower() for x in languages]

    if (
        len(languages) == 1
        and lowercase[0] == "english"
    ):
        return "ENGLISH_ONLY"

    if "english" in lowercase:
        return "MULTI_WITH_ENGLISH"

    if len(languages) == 1:
        return "NON_ENGLISH_SINGLE"

    return "NON_ENGLISH_MULTI"


df["language_group"] = (
    df["canonical_language_string_cleaned"]
    .apply(language_group)
)

language_table = pd.crosstab(
    df["language_group"],
    df["human_content_country_verdict"]
)

language_percent = (
    language_table
    .div(language_table.sum(axis=1), axis=0)
    * 100
).round(1)

print("\n=== LANGUAGE GROUP COUNTS ===")
print(language_table)

print("\n=== LANGUAGE GROUP PERCENT ===")
print(language_percent)


# ============================================================
# 10. Optional rough population projection
# ============================================================

CORE_COUNTRY_ASSIGNED = 11764

print("\n=== ROUGH POPULATION PROJECTION ===")
print("For planning only — not corrected population counts.")

for verdict in ["Correct", "Inferred", "Incorrect"]:

    k = verdict_counts[verdict]

    p, low, high = wilson_ci(k, 400)

    print(
        verdict,
        "estimate:",
        round(p * CORE_COUNTRY_ASSIGNED),
        "CI:",
        round(low * CORE_COUNTRY_ASSIGNED),
        "to",
        round(high * CORE_COUNTRY_ASSIGNED)
    )

Audit file passed integrity checks.

=== OVERALL CONTENT-COUNTRY VERDICT ===
Correct   : 128/400 =  32.0% (95% CI 27.6–36.7%)
Inferred  :  85/400 =  21.2% (95% CI 17.5–25.5%)
Incorrect : 187/400 =  46.8% (95% CI 41.9–51.6%)

Correct + Inferred: 213/400 = 53.2% (95% CI 48.4–58.1%)

=== COUNTS BY PRODUCER/CONTENT RELATIONSHIP ===
human_content_country_verdict  Correct  Incorrect  Inferred
producer_content_relationship                              
EXACT                               72        117        52
NONE                                29         23        16
PARTIAL                             27         46        14
PRODUCER_MISSING                     0          1         3

=== PERCENT BY PRODUCER/CONTENT RELATIONSHIP ===
human_content_country_verdict  Correct  Incorrect  Inferred
producer_content_relationship                              
EXACT                             29.9       48.5      21.6
NONE                              42.6       33.8      23.5
PARTIAL             